# TickPredict: train and compare the image models

Choose **life stage**, **engorgement**, or both. `testing.ipynb` runs saved checkpoints on a supplied image.

The original Penn dataset is now available under `dataset/`; local paths are detected automatically. The fixed split is 241 train / 110 development images; protected test images are never loaded by this notebook.

By default, this notebook shows the selected training recipes and available development results. All execution switches are `False`. Model comparisons use ordinary validation accuracy, while reporting balanced accuracy, every class, and stage × engorgement subgroups. These repeatedly used validation results are not independent test results.

Shared code: [original baseline trainer](scripts/train.py), [experiment trainer](scripts/experiment.py), [model definitions](scripts/modeling.py), and [comparison metrics](scripts/compare_runs.py).


In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
WORK = next(
    (folder for folder in (cwd, *cwd.parents)
     if (folder / "scripts/common.py").is_file()
     and (folder / "configs/life_stage.json").is_file()),
    None,
)
if WORK is None:
    raise RuntimeError("Open this notebook with Jupyter started in the project directory.")
if str(WORK / "scripts") not in sys.path:
    sys.path.insert(0, str(WORK / "scripts"))
print("Project:", WORK)


## 1. Choose what to train

`TRAINING_MODE = "selected"` retrains the recipes behind the current predictors, recorded in `configs/current_training.json`. Choose one task or both in `TASKS`. Every model used in an ensemble is listed explicitly.

`"baseline"` reproduces the original ResNet18 algorithm. `"experiments"` runs a comparison plan chosen by `PLAN_PATH`; set `RECIPES` to a list of names or leave `None` for all recipes. In experiment mode, `TASKS` must be supported by the selected plan.

Set only the action switches you intend to execute. Training uses the local GPU, including MPS on Apple Silicon. On another computer, the first training run may download its pinned ImageNet initialization.


In [ ]:
import json

TASKS = ["life_stage", "engorgement_level"]
TRAINING_MODE = "selected"  # Or "experiments" or "baseline"
PLAN_PATH = "configs/experiments.json"  # Or configs/life_stage_replications.json
RECIPES = None  # None = all recipes in PLAN_PATH; otherwise a list of recipe names
RUN_PREFLIGHT = False
RUN_SMOKE = False
RUN_TRAINING = False
EXPORT_PLOTS = False

if not TASKS or not set(TASKS) <= {"life_stage", "engorgement_level"} or len(set(TASKS)) != len(TASKS):
    raise ValueError("Choose life_stage, engorgement_level, or both, without duplicates.")
if TRAINING_MODE not in {"selected", "baseline", "experiments"}:
    raise ValueError("TRAINING_MODE must be selected, baseline or experiments.")
configs = {task: WORK / "configs" / f"{task}.json" for task in TASKS}
plan_path = Path(PLAN_PATH).expanduser()
if not plan_path.is_absolute():
    plan_path = WORK / plan_path
if TRAINING_MODE == "selected":
    selected_training = json.loads((WORK / "configs/current_training.json").read_text())
    selected_configs = [cfg for task in TASKS for cfg in selected_training["models"][task]]
    for config in selected_configs:
        print(f"{config['target']}: {config['name']}, {config['image_size']} px, seed {config['seed']}")
elif TRAINING_MODE == "baseline":
    for task, config in configs.items():
        print(task, json.loads(config.read_text()))
else:
    from run_experiments import planned_configs
    for config in planned_configs(plan_path, TASKS, RECIPES):
        print(f"{config['target']}: {config['name']}, {config['image_size']} px, "
              f"up to {config['epochs']} epochs, seed {config['seed']}")


## 2. Verify the data

The default local paths are `dataset/Penn_Dataset/prepared_data.csv` and `dataset/Penn Data/433 tick photos/`. You can override them with `TICKPREDICT_SOURCE_CSV` and `TICKPREDICT_IMAGE_DIR` before launching Jupyter; restart the kernel after changing the source path. The baseline initialization is `data/weights/resnet18-f37072fd.pth` (override with `TICKPREDICT_WEIGHTS`).

Enable `RUN_PREFLIGHT` to verify source/manifest/image hashes, train/dev decoding, and the original ID/hash/similarity-group separation. This preserves test assignments without opening test images. Engorgement class boundaries are not independently verified; participant-level independence is not established.


In [ ]:
if RUN_PREFLIGHT:
    from prepare import prepare
    report = prepare()
else:
    print("Preflight skipped. Configure the source paths, then set RUN_PREFLIGHT = True.")

## 3. Optional smoke tests

After a successful preflight, set `RUN_SMOKE = True` in the settings cell. A small run for each selected task checks image loading, gradients, and checkpoint writing. Its scores are not model-performance evidence.

In [ ]:
if RUN_SMOKE:
    if TRAINING_MODE == "selected":
        from experiment import run
        smoke_folders = [run(config, smoke=True) for config in selected_configs]
    elif TRAINING_MODE == "baseline":
        from train import run
        smoke_folders = {task: run(config, smoke=True, output_root=WORK / "experiments/reproductions")
                         for task, config in configs.items()}
    else:
        from run_experiments import run_plan
        smoke_folders = run_plan(plan_path, TASKS, RECIPES, execute=True, smoke=True)
else:
    print("Smoke tests skipped. Set RUN_SMOKE = True to enable them.")


## 4. Train the selected models

Enable `RUN_TRAINING` after preflight and smoke checks. Models train sequentially on the GPU; each run records its full settings, predictions, metrics, and selected checkpoint. Baseline reproductions use `experiments/reproductions/`; comparisons use `experiments/runs/`. Re-running a comparison skips completed matching configurations with the same code/data fingerprints.

The experiment trainer warms up the head, then fine-tunes the requested layers with frozen batch-normalization running statistics. Higher-resolution letterboxing and rotation preserve the full image. No new checkpoint automatically changes the explicit selection in `configs/current_models.json`.


In [ ]:
if RUN_TRAINING:
    if TRAINING_MODE == "selected":
        from experiment import run
        run_folders = [run(config) for config in selected_configs]
    elif TRAINING_MODE == "baseline":
        from train import run
        run_folders = {task: run(config, output_root=WORK / "experiments/reproductions")
                       for task, config in configs.items()}
    else:
        from run_experiments import run_plan
        run_folders = run_plan(plan_path, TASKS, RECIPES, execute=True)
else:
    print("Training skipped. Set RUN_TRAINING = True to train the selected tasks.")


## 5. Compare completed development results

This table excludes smoke, running, and failed runs. A candidate is eligible for promotion only if its accuracy is at least the pinned baseline's and no class loses more than one correct development prediction. Detailed per-class, subgroup, and confidence metrics are available in `results`.

Strictly above 95% requires at least 105 correct predictions out of 110 development images. The protected test remains unopened until a final recipe is selected; passing this development threshold alone does not establish the final target.


In [ ]:
from compare_runs import collect_results
from IPython.display import Markdown, display

results = [row for row in collect_results(WORK) if row["target"] in TASKS]
lines = ["| Task | Recipe / run | Correct | Accuracy | Balanced | Eligible |",
         "|---|---|---:|---:|---:|---|"]
for row in sorted(results, key=lambda r: (r["target"], -r["validation_accuracy"])):
    lines.append(f"| {row['target']} | {Path(row['run']).name} | {row['correct']}/{row['validation_n']} | "
                 f"{row['validation_accuracy']:.1%} | {row['validation_balanced_accuracy']:.1%} | {row['promotion_eligible']} |")
display(Markdown("\n".join(lines)))

if EXPORT_PLOTS:
    from plot_run import plot_run
    from IPython.display import Image
    for task in TASKS:
        eligible = [r for r in results if r["target"] == task and r["promotion_eligible"]]
        if eligible:
            best = max(eligible, key=lambda r: (r["validation_accuracy"], r["validation_balanced_accuracy"], -r["validation_nll"]))
            display(Image(filename=str(plot_run(WORK / best["run"]))))
